In [476]:
%pip install joblib

In [477]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder,OneHotEncoder, LabelEncoder # preprocessing
from sklearn.metrics import silhouette_score, adjusted_rand_score, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer # transform columns
import time # to calculate execution time
from scipy.optimize import linear_sum_assignment # match cluster labels to true class labels optimally
from joblib import Parallel, delayed
import itertools # to create combinations of hyperparameters




### Openml
In Python, OpenML is mainly used to discover, download, and share ML datasets, tasks, and results—super handy for experiments, benchmarking, and learning ML properly.

In [478]:
%pip install openml



In [479]:
import openml

### Download the dataset from openl using dataset id

In [480]:
def download_dataset(dataset_id):
    dataset = openml.datasets.get_dataset(dataset_id)
    X,y, categorical_indicator, attribute_names = dataset.get_data(
    target=dataset.default_target_attribute)
    return X,y, categorical_indicator, attribute_names


### Prepare data
Define numerical and categorical columns based on the categorical_indicator

In [481]:
# Define columns types
def define_column_types(X):
    cat_columns = X.columns[np.array(categorical_indicator)==True]
    num_columns = X.columns[np.array(categorical_indicator)==False]
    return list(num_columns), list(cat_columns)

# Pre-processing
Numeric variables --> scale
Ordinal variables -->  Ordinal encoding
Nominal categorical variables -->  Onehot encoding
Binary(already 0 and 1) -->  onehot encoding 


# Define parameters

In [482]:
def define_parameters(algorithm):
    if algorithm == "KMeans":
        param_grid = {
        "n_clusters": [2, 3, 4, 5, 6],
        "init": ["k-means++", "random"],
        "n_init": [10, 20],
        "max_iter": [300, 500]
        }

    if algorithm == "AgglomerativeClustering":
        param_grid ={
        "n_clusters": [2, 3, 4,5,6],
        "linkage": ["single","average", "complete"],
        "metric": ["euclidean", "manhattan"] 
        }

    if algorithm == "Ward":
        param_grid ={
        "n_clusters": [2, 3, 4,5,6],
        "linkage": ["ward"], # only ward can be used
        "metric": ["euclidean"] # only euclidean can be used
        }


    param_names = list(param_grid.keys())
    param_combinations =list(itertools.product(    
        *(param_grid[param_name] for param_name in param_names))
        )
    return  param_grid, param_combinations, param_names


# Define model

In [483]:
def evaluate_performance(params):
    start = time.time()

    keys = param_names
    params_dict = dict(zip(keys, params))
    #print(params_dict)

    if algorithm == "KMeans":
        pipe = Pipeline([
            ("preprocess" , preprocessor),
            (algorithm, algorithms[algorithm](**params_dict, random_state=42))
            ])
    else:
        pipe = Pipeline([
            ("preprocess" , preprocessor),
            (algorithm, algorithms[algorithm](**params_dict))
            ])
    
    
    y_pred = pipe.fit_predict(X)

    # silhouette_score
    silhouette = silhouette_score(X, y_pred)

    # adjusted_rand_score
    ari = adjusted_rand_score(y_enc, y_pred) # no need to do label alignment

    # f1_score
    # Align cluster labels to true labels using Hungarian algorithm
    def align_cluster_labels(y_enc, y_pred):
        cm = confusion_matrix(y_enc, y_pred)  
        row_ind, col_ind = linear_sum_assignment(-cm) # # Maximize diagonal → minimize negative
        mapping = {col: row for row, col in zip(row_ind, col_ind)} 
        return np.array([mapping[label] for label in y_pred])

    y_pred_aligned = align_cluster_labels(y_enc, y_pred)
    f1 = f1_score(y_enc, y_pred_aligned, average="macro") # Computes F1 per class, takes an unweighted mean, treats all clusters/classes equally
    
   
    execution_time = float(time.time() - start)
    #params_dict["inertia"] = pipe.named_steps[algorithm].inertia_
    params_dict["silhouette_score"] = silhouette
    params_dict["adjusted_rand_score"] = ari
    params_dict["f1_score"] = f1
    params_dict["execution_time"] = execution_time
    

    return params_dict



### Joblib 
joblib is a Python library mainly used in ML for saving models, fast loading, and parallel processing

n_jobs answers “how many things can run in parallel? n_jobs does NOT say threads or processes.
Threading is how parallelism is done. Threading is a backend choice in joblib.
backend="threading"   # threads
backend="loky"        # processes (default)

### Create lists of openml datasets and datasets to be analyzed

In [484]:
dataset_names  = ["iris", "wine"] # datasets to be analyzed

datasets = openml.datasets.list_datasets(output_format="dataframe") # openml datasets

In [485]:
algorithms = {
    "KMeans": KMeans, 
    "AgglomerativeClustering": AgglomerativeClustering,
    "Ward": AgglomerativeClustering
}

results_all = pd.DataFrame()

for algorithm in algorithms:
    for dataset_name in dataset_names:
        dataset_id = int(datasets.loc[datasets["name"] == dataset_name, "did"].values[0])
        # Download dataset
        X,y, categorical_indicator, attribute_names = download_dataset(dataset_id)

        # define trypes of columns in X
        num_columns, cat_columns = define_column_types(X) 

        # preprocess y
        label_enc = LabelEncoder()
        y_enc = label_enc.fit_transform(y)

        # define preprocessor for X
        preprocessor = ColumnTransformer(
        transformers = [
            ("num", MinMaxScaler(), num_columns),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns),
        ]
        )
        
        # define parameter grid
        param_grid, param_combinations, param_names = define_parameters(algorithm)
    
        # evaluate performance of each algorithm
        results = Parallel(n_jobs=-1, verbose=10)(
        delayed(evaluate_performance)(params) for params in param_combinations
        )
        results_df = pd.DataFrame(results)
        results_df.insert(0,"dataset", dataset_name)
        results_df.insert(1,"algorithm", algorithm)
        results_all = pd.concat([results_all, results_df], ignore_index=True)
        print(results_all[results_all["dataset"]=="wine"])
        
results_all.to_csv("results_all.csv")

            
    
   

 


    


    



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1440s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done  14 out of  40 | elapsed:    0.2s remaining:    0.4s
[Parallel(n_jobs=-1)]: Done  19 out of  40 | elapsed:    0.3s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done  24 out of  40 | elapsed:    0.3s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  29 out of  40 | elapsed:    0.3s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  34 out of  40 | elapsed:    0.3s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  40 out of  40 | elapsed:    0.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.


Empty DataFrame
Columns: [dataset, algorithm, n_clusters, init, n_init, max_iter, silhouette_score, adjusted_rand_score, f1_score, execution_time]
Index: []


[Parallel(n_jobs=-1)]: Batch computation too fast (0.0826s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done  14 out of  40 | elapsed:    0.1s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done  19 out of  40 | elapsed:    0.1s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done  24 out of  40 | elapsed:    0.2s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  29 out of  40 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  34 out of  40 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  40 out of  40 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.1882s.) Setting batch_size=2.


   dataset algorithm  n_clusters       init  n_init  max_iter  \
40    wine    KMeans           2  k-means++      10       300   
41    wine    KMeans           2  k-means++      10       500   
42    wine    KMeans           2  k-means++      20       300   
43    wine    KMeans           2  k-means++      20       500   
44    wine    KMeans           2     random      10       300   
45    wine    KMeans           2     random      10       500   
46    wine    KMeans           2     random      20       300   
47    wine    KMeans           2     random      20       500   
48    wine    KMeans           3  k-means++      10       300   
49    wine    KMeans           3  k-means++      10       500   
50    wine    KMeans           3  k-means++      20       300   
51    wine    KMeans           3  k-means++      20       500   
52    wine    KMeans           3     random      10       300   
53    wine    KMeans           3     random      10       500   
54    wine    KMeans     

[Parallel(n_jobs=-1)]: Done   3 out of  30 | elapsed:    0.1s remaining:    1.7s
[Parallel(n_jobs=-1)]: Done   7 out of  30 | elapsed:    0.1s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done  11 out of  30 | elapsed:    0.2s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done  15 out of  30 | elapsed:    0.2s remaining:    0.2s
[Parallel(n_jobs=-1)]: Done  19 out of  30 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  23 out of  30 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  27 out of  30 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    0.2s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.0375s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   3 out of  30 | elapsed:    0.0s remaining:    0.5s


   dataset algorithm  n_clusters       init  n_init  max_iter  \
40    wine    KMeans           2  k-means++    10.0     300.0   
41    wine    KMeans           2  k-means++    10.0     500.0   
42    wine    KMeans           2  k-means++    20.0     300.0   
43    wine    KMeans           2  k-means++    20.0     500.0   
44    wine    KMeans           2     random    10.0     300.0   
45    wine    KMeans           2     random    10.0     500.0   
46    wine    KMeans           2     random    20.0     300.0   
47    wine    KMeans           2     random    20.0     500.0   
48    wine    KMeans           3  k-means++    10.0     300.0   
49    wine    KMeans           3  k-means++    10.0     500.0   
50    wine    KMeans           3  k-means++    20.0     300.0   
51    wine    KMeans           3  k-means++    20.0     500.0   
52    wine    KMeans           3     random    10.0     300.0   
53    wine    KMeans           3     random    10.0     500.0   
54    wine    KMeans     

[Parallel(n_jobs=-1)]: Done   7 out of  30 | elapsed:    0.0s remaining:    0.3s
[Parallel(n_jobs=-1)]: Done  11 out of  30 | elapsed:    0.0s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  15 out of  30 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  19 out of  30 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  23 out of  30 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  27 out of  30 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Batch computation too fast (0.0292s.) Setting batch_size=2.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   5 out of  

    dataset                algorithm  n_clusters       init  n_init  max_iter  \
40     wine                   KMeans           2  k-means++    10.0     300.0   
41     wine                   KMeans           2  k-means++    10.0     500.0   
42     wine                   KMeans           2  k-means++    20.0     300.0   
43     wine                   KMeans           2  k-means++    20.0     500.0   
44     wine                   KMeans           2     random    10.0     300.0   
..      ...                      ...         ...        ...     ...       ...   
135    wine  AgglomerativeClustering           6        NaN     NaN       NaN   
136    wine  AgglomerativeClustering           6        NaN     NaN       NaN   
137    wine  AgglomerativeClustering           6        NaN     NaN       NaN   
138    wine  AgglomerativeClustering           6        NaN     NaN       NaN   
139    wine  AgglomerativeClustering           6        NaN     NaN       NaN   

     silhouette_score  adju

[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    0.0s finished
